In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

# Define the Margalla Fault using its known Longitude/Latitude path north of the city
fault_coords = [(72.85, 33.73), (73.05, 33.77), (73.20, 33.76)]
fault_line = LineString(fault_coords)

# Create a spatial dataframe for the fault line using standard GPS coordinates (EPSG:4326)
fault_gdf = gpd.GeoDataFrame(geometry=[fault_line], crs="EPSG:4326")

# CRITICAL STEP: Convert to a metric coordinate system (UTM Zone 43N) to measure in meters
fault_metric = fault_gdf.to_crs("EPSG:32643")

# Generate expanding blast-radius buffers (distances are now strictly in meters)
buffer_2km = fault_metric.buffer(2000)
buffer_5km = fault_metric.buffer(5000)
buffer_10km = fault_metric.buffer(10000)

# Convert these raw shapes back into GeoDataFrames with assigned risk levels
zone1 = gpd.GeoDataFrame({'risk_level': ['Extreme'], 'risk_weight': [3]}, geometry=buffer_2km, crs="EPSG:32643")
zone2 = gpd.GeoDataFrame({'risk_level': ['High'], 'risk_weight': [2]}, geometry=buffer_5km, crs="EPSG:32643")
zone3 = gpd.GeoDataFrame({'risk_level': ['Moderate'], 'risk_weight': [1]}, geometry=buffer_10km, crs="EPSG:32643")

# Punch out the centers to create distinct 'donut' rings (so zones don't overlap)
ring_5km = gpd.overlay(zone2, zone1, how='difference')
ring_10km = gpd.overlay(zone3, zone2, how='difference')

# Combine all three distinct zones into one master table
final_buffers = pd.concat([zone1, ring_5km, ring_10km])

# Project back to standard GPS coordinates to match your building files
final_buffers = final_buffers.to_crs("EPSG:4326")

# Save the final seismic zones
final_buffers.to_file('data/Margalla_Fault_Buffers.gpkg', driver='GPKG')

print("Fault buffers generated successfully in metric space.")

Fault buffers generated successfully in metric space.
